[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/triality/triality.ipynb)

# Spin(8) Triality

This example computes Spin(8) triality, the symmetry that links the vector space of Spin(8) and its two chiral spinor spaces through the octonion product. Each of the three spaces is modelled as eight-dimensional real octonion space, and rotating a product of two octonions agrees with multiplying the two factors after turning each by a companion rotation of its own, even where none of the three rotations preserves the product on its own. Requiring all three rotations to coincide selects the symmetries of the octonion product itself, the exceptional group G2.

An octonion is modelled as a scalar plus a vector in seven-dimensional negative-definite space, and multiplication as one bilinear map `multiply = ((Octonion * Octonion) * (1 - fano)).cast(Octonion)` built from the trivector sum `fano` of the seven oriented lines of the Fano plane. Minus the trace of a composition pairs skew maps, so a single `.solve` against the left and right multiplication families splits every infinitesimal rotation into two vector multiplications and a derivation, assembling the three companions satisfying `output(multiply) == multiply(left, Octonion) + multiply(Octonion, right)` and isolating G2 as the nullspace of the offset form.

In Lie-algebra and matrix representation notation, the vector representation and the two chiral half-spin representations of $\mathfrak{so}(8)$ are written through an explicit family of eight $8 \times 8$ real Clifford matrices $\Gamma_1, \dots, \Gamma_8$ built from Kronecker products of $2 \times 2$ blocks, with generators $E_{ij} - E_{ji}$ on vectors and $\tfrac{1}{4}(\Gamma_i \Gamma_j^\top - \Gamma_j \Gamma_i^\top)$ or $\tfrac{1}{4}(\Gamma_i^\top \Gamma_j - \Gamma_j^\top \Gamma_i)$ on the two spinor spaces, linked across the $D_4$ Dynkin diagram ([Baez, *The Octonions*](https://math.ucr.edu/home/baez/octonions/node7.html)) by the infinitesimal triality identity $\theta_1(xy) = \theta_2(x)\,y + x\,\theta_3(y)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
import numpy as np

from numga import Algebra, NumpyContext, stack

np.set_printoptions(precision=4, suppress=True)

ga = Algebra("a-b-c-d-e-f-g-")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Vector = ga.gatype.vector()                                                  # Vector+
Octonion = ga.gatype.from_grades([0, 1])                                     # Scalar+ + Vector+
Generator = ga.gatype.from_grades([1, 2])                                    # Vector+ + Bivector+
OctonionMap = ga.gatype((Octonion, Octonion))                                # Octonion+ <- Octonion+
OffsetMap = ga.gatype((Vector, Generator))                                   # Vector+ <- Generator+

## Octonion multiplication

An octonion occupies the scalar and vector grades of the algebra, and seven oriented triples specify its multiplication table. Clifford conjugation negates the vector part, compensating for its negative square so that `value.symmetric_conjugate_product()` measures squared Euclidean length. Octonion multiplication preserves this positive length, though its brackets matter.

In [ ]:
# Each oriented triple closes under the product, `a` times `b` giving `c`; summed, the seven triples are the
# oriented lines of the Fano plane, one trivector that carries the whole multiplication table.
cycles = stack([mv.abc, mv.ade, mv.agf, mv.bdf, mv.beg, mv.cdg, mv.cfe])     # [7] Trivector+
fano = cycles.sum(axis=0)                                                    # [] Trivector+
multiply = ((Octonion * Octonion) * (1 - fano)).cast(Octonion)               # [] Octonion+ <- (Octonion+, Octonion+)

## Three linked rotations

Turning the plane spanned by the scalar and a unit vector direction fixes the six orthogonal vector directions. Multiplication on the left and on the right by the half-angle exponential `(direction * (angles / 2)).exp()` gives the two companion rotations, each turning four planes through half the angle so that `output_rotation(multiply)` agrees with `multiply(left_rotation, right_rotation)`.

In octonion algebra notation, with $u = \exp(\tfrac{1}{2}\theta\,e_1)$ that unit octonion, this agreement is the Moufang identity $u(xy)u = (ux)(yu)$.

In [ ]:
from examples.math.triality.scenarios import angle

direction = mv.a                                                             # [] Vector+
first = 1 + mv.b                                                             # [] Octonion+
second = mv.c + mv.d                                                         # [] Octonion+

In [ ]:
def finite_actions(direction: Vector,
                   angles: np.ndarray) -> tuple[OctonionMap, OctonionMap, OctonionMap]:
    """A plane rotation and its two companions (`[...] Octonion+ <- Octonion+`) for a unit vector direction."""
    # Powers of a single vector stay scalar plus vector; narrow the generic exponential.
    unit = (direction * (angles / 2)).exp().cast(Octonion)                   # [...] Octonion+
    left = multiply(unit, Octonion)                                          # [...] Octonion+ <- Octonion+
    right = multiply(Octonion, unit)                                         # [...] Octonion+ <- Octonion+
    output = multiply(unit, multiply(Octonion, unit))                        # [...] Octonion+ <- Octonion+
    return output, left, right


output_rotation, left_rotation, right_rotation = finite_actions(direction, angle)  # [] Octonion+ <- Octonion+
for name, action in (("Output rotation", output_rotation),
                     ("First-factor rotation", left_rotation),
                     ("Second-factor rotation", right_rotation)):
    print(name)
    print(action.coefficient_table())

rotated_product = output_rotation(multiply(first, second))                   # [] Octonion+
product_of_rotated = multiply(left_rotation(first), right_rotation(second))  # [] Octonion+
print("Rotate the product")
print(rotated_product.coefficient_table())
print("Multiply the rotated factors")
print(product_of_rotated.coefficient_table())

## One turn and two turns

After one full turn the output action returns to the identity while both factor actions become minus the identity, whose opposite signs cancel in the product; two full turns return all three to the identity. Identifying the output space with the vector representation of Spin(8) makes the two factor spaces its two eight-dimensional chiral spinor representations.

In [ ]:
from examples.math.triality.scenarios import action_names, turn_angles, turn_names

In [ ]:
output_returns, left_returns, right_returns = finite_actions(direction, turn_angles)  # [3] Octonion+ <- Octonion+

# At full turns each map is a scalar multiple of the identity, read by dividing its trace by 8.
identity_trace = Octonion.trace()                                            # [] Scalar+
return_factors = stack([output_returns.trace(), left_returns.trace(),
                        right_returns.trace()]) / identity_trace             # [3, 3] Scalar+
print(turn_names)
for index, name in enumerate(action_names):
    print(name, return_factors[index].to_array())

## Multiplication offsets of general rotations

Every infinitesimal rotation of the eight components splits into left multiplication by a vector, right multiplication by a vector, and a common part obeying the product rule, which is called a derivation. Minus the trace of a composition pairs skew maps positively, so solving the trace Gram system of the two multiplication families extracts both vector offsets.

In [ ]:
# Bivectors turn the seven vector components; vectors turn them into the scalar.
turn = (Generator.restrict.bivector().commutator(Octonion)
        + Generator.restrict.vector().anticommutator(Octonion))              # [] Octonion+ <- (Generator+, Octonion+)


def multiplication_components() -> tuple[OffsetMap, OffsetMap]:
    """The left and right multiplication offset maps (`Vector+ <- Generator+`) of an infinitesimal rotation."""
    left_multiply = multiply(Vector, Octonion)                               # [] Octonion+ <- (Vector+, Octonion+)
    # Grade involution reverses the order of an octonion product, so both maps take the multiplying vector first.
    right_multiply = -multiply(Vector, Octonion.involute()).involute()       # [] Octonion+ <- (Vector+, Octonion+)
    multiplications = stack([left_multiply, right_multiply])                 # [2] Octonion+ <- (Vector+, Octonion+)

    # Minus the trace of a composition is the positive pairing of two skew maps.
    projection_form = (
        -multiplications[:, None](Vector, multiplications[None, :]).trace(0, 3)
    ).field(1, 2)                                                            # [] (Scalar+ <- (Vector+, Vector+))[2 <- 2]
    overlap = (-turn(Generator, multiplications).trace(0, 3)).field(2)       # [] (Scalar+ <- (Generator+, Vector+))[2]
    left_component, right_component = projection_form.solve(overlap).batch()  # [] Vector+ <- Generator+, Vector+ <- Generator+
    return left_component, right_component


left_component, right_component = multiplication_components()

## Common G2 derivations

The squared lengths of the two multiplication offsets define a nonnegative quadratic form on generators measuring departure from a derivation. Its zero modes isolate the 14 common generators of G2, while all 28 eigendirections supply an orthonormal frame of Spin(8) generators.

In [ ]:
from examples.math.triality.scenarios import tolerance

In [ ]:
def action_modes(left_component: OffsetMap,
                 right_component: OffsetMap) -> tuple[Scalar, Generator]:
    """A complete frame of 28 rotation generators (`[28] Generator+`), ordered by squared multiplication offsets (`[28] Scalar+`)."""
    constraint = (left_component.symmetric_conjugate_product()
                  + right_component.symmetric_conjugate_product())           # [] Scalar+ <- (Generator+, Generator+)
    # Clifford conjugation gives a positive pairing on both grades of a generator.
    metric = Generator.clifford_conjugate().scalar_product(Generator)        # [] Scalar+ <- (Generator+, Generator+)
    return constraint.eigh(metric)


penalties, generators = action_modes(left_component, right_component)        # [28] Scalar+, Generator+
common_generators = generators[penalties < tolerance]                        # [14] Generator+

print(f"Triality generators: {len(generators)}")
print(f"Common generators: {len(common_generators)}")

## Linked generator actions

A derivation acts identically on both factors and their product, whereas redistributing the left and right multiplication offsets supplies the two factor companions for every generator of Spin(8). Because the three infinitesimal actions satisfy the product rule, exponentiating them together preserves the finite octonion product.

In [ ]:
def linked_actions(generators: Generator, left_component: OffsetMap,
                   right_component: OffsetMap) -> tuple[OctonionMap, OctonionMap, OctonionMap]:
    """The product action and its two factor actions (`[...] Octonion+ <- Octonion+`), obeying the product rule."""
    output = turn(generators, Octonion)                                      # [...] Octonion+ <- Octonion+
    left_offset = left_component(generators)                                 # [...] Vector+
    right_offset = right_component(generators)                               # [...] Vector+
    derivation = (output - multiply(left_offset, Octonion)
                  - multiply(Octonion, right_offset))                        # [...] Octonion+ <- Octonion+
    left = (derivation + multiply(left_offset, Octonion)
            + multiply(Octonion, left_offset - right_offset))                # [...] Octonion+ <- Octonion+
    right = (derivation + multiply(right_offset - left_offset, Octonion)
             + multiply(Octonion, right_offset))                             # [...] Octonion+ <- Octonion+
    return output, left, right


output, left, right = linked_actions(generators, left_component, right_component)  # [28] Octonion+ <- Octonion+

The 28 linked generators span the Lie algebra of Spin(8), while the 14 common derivations span the Lie algebra of G2, where all three actions agree and outer automorphisms of Spin(8) fix every element.

In [ ]:
# checks
# Finite plane rotation agrees with multiplying rotated factors, and returns after two turns.
assert (rotated_product - product_of_rotated).symmetric_conjugate_product() < 1e-8
expected_returns = np.array([[1, 1, 1], [1, -1, 1], [1, -1, 1]])
assert ((return_factors - expected_returns).abs() < 1e-8).all()

# Every generator's three actions obey the product rule, and fourteen generators are common to all three.
product_rule = (output(multiply) - multiply(left, Octonion)
                - multiply(Octonion, right))                                 # [28] Octonion+ <- (Octonion+, Octonion+)
product_rule_defect = product_rule.symmetric_conjugate_product().contract(1, 3).contract(1, 2)
assert (product_rule_defect.abs() < 1e-8).all()
assert len(generators) == 28 and len(common_generators) == 14